In [ ]:
from pathlib import Path

import pandas as pd
from google.colab import files

raw_dir = Path("/content/week3_project/data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

filename = "hotel_bookings.csv"
csv_path = raw_dir / filename

if not csv_path.exists():
    uploaded = files.upload()
    if filename not in uploaded:
        raise ValueError(f"Please upload the file named {filename}.")
    csv_path.write_bytes(uploaded[filename])

bookings = pd.read_csv(csv_path)
print(f"{len(bookings):,} records and {bookings.shape[1]} columns")

Saving hotel_bookings.csv to hotel_bookings.csv
119,390 records and 32 columns


In [ ]:
display(pd.crosstab(
    bookings["hotel"],
    bookings["is_canceled"],
    margins=True,
))
print("Missing target values:", bookings["is_canceled"].isna().sum())

preview_columns = [
    "hotel", "arrival_date_year", "arrival_date_month",
    "lead_time", "is_canceled",
]

for hotel in ["City Hotel", "Resort Hotel"]:
    subset = bookings.loc[bookings["hotel"].eq(hotel), preview_columns]
    print(f"\n{hotel}: {len(subset):,} records")
    display(subset.head(2))

missing_counts = bookings.isna().sum()
display(missing_counts[missing_counts.gt(0)].sort_values(ascending=False))

is_canceled,0,1,All
hotel,,,
City Hotel,46228,33102,79330
Resort Hotel,28938,11122,40060
All,75166,44224,119390


Missing target values: 0

City Hotel: 79,330 records


,hotel,arrival_date_year,arrival_date_month,lead_time,is_canceled
40060,City Hotel,2015,July,6,0
40061,City Hotel,2015,July,88,1



Resort Hotel: 40,060 records


,hotel,arrival_date_year,arrival_date_month,lead_time,is_canceled
0,Resort Hotel,2015,July,342,0
1,Resort Hotel,2015,July,737,0


,0
company,112593
agent,16340
country,488
children,4


In [ ]:
display(pd.crosstab(
    bookings["reservation_status"],
    bookings["is_canceled"],
    dropna=False,
))

arrival_dates = pd.to_datetime(
    bookings["arrival_date_year"].astype(str)
    + " " + bookings["arrival_date_month"].astype(str)
    + " " + bookings["arrival_date_day_of_month"].astype(str),
    format="%Y %B %d",
    errors="coerce",
)

print("Earliest arrival:", arrival_dates.min())
print("Latest arrival:", arrival_dates.max())
print("Unparseable arrival dates:", arrival_dates.isna().sum())

is_canceled,0,1
reservation_status,,
Canceled,0,43017
Check-Out,75166,0
No-Show,0,1207


Earliest arrival: 2015-07-01 00:00:00
Latest arrival: 2017-08-31 00:00:00
Unparseable arrival dates: 0


## Lab 3 Part B — Data Preparation
This preparation step keeps all booking rows and selects a small set of columns for NullStay. The initial candidate inputs are hotel type, lead time, and the booked arrival month. Their availability at the intended prediction point will be reviewed before training. The full arrival date is retained for historical filtering and split design, and `is_canceled` is retained as the outcome label. In this dataset, label `1` covers cancellations and no-shows.

The preparation step excludes final reservation fields such as `reservation_status` and `reservation_status_date`. It preserves repeated rows because identical values can describe separate bookings. This step checks and selects fields without fitting any preprocessing. Any preprocessing that learns from the data will be fitted on the training partition.

In [4]:
import hashlib
import json

candidate_features = ["hotel", "lead_time", "arrival_date_month"]
target = "is_canceled"

prepared = bookings[candidate_features + [target]].copy()
prepared.insert(0, "arrival_date", arrival_dates)

assert len(bookings) > 0, "The source file contains no bookings."
assert len(prepared) == len(bookings), "The row count changed."
assert prepared[target].equals(bookings[target]), "Target values changed."
assert set(prepared[target].unique()) == {0, 1}, "Check the target labels."
assert prepared.isna().sum().sum() == 0, "Check missing values in retained fields."
assert prepared["lead_time"].ge(0).all(), "Check negative lead times."
assert set(prepared["hotel"].unique()) == {"City Hotel", "Resort Hotel"}

comparison = pd.DataFrame({
    "Original": {
        "Rows": len(bookings),
        "Columns": bookings.shape[1],
        "Missing cells": int(bookings.isna().sum().sum()),
        "Target 1 rows": int(bookings[target].eq(1).sum()),
    },
    "Prepared": {
        "Rows": len(prepared),
        "Columns": prepared.shape[1],
        "Missing cells": int(prepared.isna().sum().sum()),
        "Target 1 rows": int(prepared[target].eq(1).sum()),
    },
})
display(comparison)
display(prepared.head())

data_dir = csv_path.parent.parent
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

prepared_path = processed_dir / "prepared_table.csv"
prepared.to_csv(prepared_path, index=False, date_format="%Y-%m-%d")

manifest = {
    "source_dataset": "Hotel Booking Demand",
    "source_file": f"raw/{csv_path.name}",
    "source_sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(),
    "output_file": "processed/prepared_table.csv",
    "output_sha256": hashlib.sha256(prepared_path.read_bytes()).hexdigest(),
    "candidate_model_inputs": candidate_features,
    "date_for_filters_and_split_design": "arrival_date",
    "target": target,
    "positive_label_meaning": "Cancellation or no-show",
    "comparison": comparison.to_dict(),
}

manifest_path = data_dir / "week3_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved:", prepared_path)
print("Saved:", manifest_path)

,Original,Prepared
Rows,119390,119390
Columns,32,5
Missing cells,129425,0
Target 1 rows,44224,44224


,arrival_date,hotel,lead_time,arrival_date_month,is_canceled
0,2015-07-01,Resort Hotel,342,July,0
1,2015-07-01,Resort Hotel,737,July,0
2,2015-07-01,Resort Hotel,7,July,0
3,2015-07-01,Resort Hotel,13,July,0
4,2015-07-01,Resort Hotel,14,July,0


Saved: /content/week3_project/data/processed/prepared_table.csv
Saved: /content/week3_project/data/week3_manifest.json


### Part B Findings

| Measure | Original table | Prepared table |
|---|---:|---:|
| Booking records | 119,390 | 119,390 |
| Columns | 32 | 5 |
| Missing cells | 129,425 | 0 |
| Records labelled cancellation or no-show | 44,224 | 44,224 |

The prepared table retains `hotel`, `lead_time`, and `arrival_date_month` as candidate model inputs. It also includes the derived `arrival_date` for historical filtering and split planning, and `is_canceled` as the target. The prepared table has zero missing cells because the columns containing missing values were excluded. No values were filled and no bookings were removed.

The validation checks confirmed that the retained fields were complete, lead times were nonnegative, and every target value matched the original. The manifest records the input and output checksums so the files can be compared on a later run.

In [5]:
import hashlib
import json
from pathlib import Path

import pandas as pd


def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def run_tabular_pipeline(source_path, output_dir):
    source_path = Path(source_path)
    output_dir = Path(output_dir)

    # Ingest and inspect
    source_hash = sha256_file(source_path)
    raw = pd.read_csv(source_path)

    candidate_inputs = ["hotel", "lead_time", "arrival_date_month"]
    target = "is_canceled"
    required_columns = candidate_inputs + [
        target, "arrival_date_year", "arrival_date_day_of_month"
    ]

    missing_columns = sorted(set(required_columns) - set(raw.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")
    if raw.empty:
        raise ValueError("The source CSV contains no bookings.")

    # Prepare
    arrival_date = pd.to_datetime(
        raw["arrival_date_year"].astype(str)
        + " " + raw["arrival_date_month"].astype(str)
        + " " + raw["arrival_date_day_of_month"].astype(str),
        format="%Y %B %d",
        errors="coerce",
    )
    prepared = raw[candidate_inputs + [target]].copy()
    prepared.insert(0, "arrival_date", arrival_date)

    # Validate before writing
    if prepared.isna().any().any():
        missing = prepared.isna().sum()
        raise ValueError(f"Missing or invalid retained values: {missing[missing.gt(0)].to_dict()}")
    if not pd.api.types.is_numeric_dtype(prepared["lead_time"]):
        raise ValueError("Lead times must be numeric.")
    if not prepared["lead_time"].ge(0).all():
        raise ValueError("Lead times must be nonnegative.")
    if set(prepared["hotel"].unique()) != {"City Hotel", "Resort Hotel"}:
        raise ValueError("Check the hotel groups in the source CSV.")
    if set(prepared[target].unique()) != {0, 1}:
        raise ValueError("Expected both binary target labels, 0 and 1.")
    if len(prepared) != len(raw) or not prepared[target].equals(raw[target]):
        raise ValueError("The row count or target values changed.")

    # Store the table and record its lineage
    output_dir.mkdir(parents=True, exist_ok=True)
    table_path = output_dir / "prepared_table.csv"
    prepared.to_csv(table_path, index=False, date_format="%Y-%m-%d")

    manifest = {
        "pipeline_version": "v1",
        "source_dataset": "Hotel Booking Demand",
        "source_file": source_path.name,
        "source_sha256": source_hash,
        "output_file": table_path.name,
        "output_sha256": sha256_file(table_path),
        "source_rows": len(raw),
        "source_columns": raw.shape[1],
        "prepared_rows": len(prepared),
        "prepared_columns": list(prepared.columns),
        "positive_labels": int(prepared[target].eq(1).sum()),
        "positive_label_meaning": "Cancellation or no-show",
        "candidate_inputs": candidate_inputs,
        "date_for_filters_and_split_design": "arrival_date",
        "target": target,
    }
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest

In [6]:
pipeline_dir = csv_path.parent.parent / "pipeline_v1" / "tabular"

first_run = run_tabular_pipeline(csv_path, pipeline_dir)

print("Prepared rows:", first_run["prepared_rows"])
print("Prepared columns:", len(first_run["prepared_columns"]))
print("Positive labels:", first_run["positive_labels"])
print("Output SHA-256:", first_run["output_sha256"])
print("Saved files in:", pipeline_dir)

Prepared rows: 119390
Prepared columns: 5
Positive labels: 44224
Output SHA-256: 7be5eaba12d514ccc33e041b0496ec1fb6b6740e451426346d7bc4be5cedef18
Saved files in: /content/week3_project/data/pipeline_v1/tabular


In [7]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as rerun_dir:
    second_run = run_tabular_pipeline(csv_path, rerun_dir)

assert first_run["source_sha256"] == second_run["source_sha256"], (
    "The two runs used different source files."
)
assert first_run["output_sha256"] == second_run["output_sha256"], (
    "The prepared CSV changed between runs."
)

print("First output SHA-256: ", first_run["output_sha256"])
print("Second output SHA-256:", second_run["output_sha256"])
print("PASS: Both runs produced identical prepared CSV files.")

First output SHA-256:  7be5eaba12d514ccc33e041b0496ec1fb6b6740e451426346d7bc4be5cedef18
Second output SHA-256: 7be5eaba12d514ccc33e041b0496ec1fb6b6740e451426346d7bc4be5cedef18
PASS: Both runs produced identical prepared CSV files.
